# Protein

## Setup

In [ ]:
#| default_exp core.protein

In [ ]:
#| export 
import pandas as pd
import requests
from functools import lru_cache

## Uniprot sequence

In [ ]:
#| export
@lru_cache()
def get_uniprot_seq(uniprot_id):
    "Queries the UniProt database to retrieve the protein sequence for a given UniProt ID."
    
    url = f"https://www.uniprot.org/uniprot/{uniprot_id}.fasta"
    response = requests.get(url)

    # Check if the request was successful (status code 200)
    if response.status_code == 200:
        data = response.text
        # The sequence starts after the first line, which is a description
        sequence = ''.join(data.split('\n')[1:]).strip()
        return sequence
    else:
        return f"Error: Unable to retrieve sequence for UniProt ID {uniprot_id}. Status code: {response.status_code}"

In [ ]:
get_uniprot_seq('P04626')

'MELAALCRWGLLLALLPPGAASTQVCTGTDMKLRLPASPETHLDMLRHLYQGCQVVQGNLELTYLPTNASLSFLQDIQEVQGYVLIAHNQVRQVPLQRLRIVRGTQLFEDNYALAVLDNGDPLNNTTPVTGASPGGLRELQLRSLTEILKGGVLIQRNPQLCYQDTILWKDIFHKNNQLALTLIDTNRSRACHPCSPMCKGSRCWGESSEDCQSLTRTVCAGGCARCKGPLPTDCCHEQCAAGCTGPKHSDCLACLHFNHSGICELHCPALVTYNTDTFESMPNPEGRYTFGASCVTACPYNYLSTDVGSCTLVCPLHNQEVTAEDGTQRCEKCSKPCARVCYGLGMEHLREVRAVTSANIQEFAGCKKIFGSLAFLPESFDGDPASNTAPLQPEQLQVFETLEEITGYLYISAWPDSLPDLSVFQNLQVIRGRILHNGAYSLTLQGLGISWLGLRSLRELGSGLALIHHNTHLCFVHTVPWDQLFRNPHQALLHTANRPEDECVGEGLACHQLCARGHCWGPGPTQCVNCSQFLRGQECVEECRVLQGLPREYVNARHCLPCHPECQPQNGSVTCFGPEADQCVACAHYKDPPFCVARCPSGVKPDLSYMPIWKFPDEEGACQPCPINCTHSCVDLDDKGCPAEQRASPLTSIISAVVGILLVVVLGVVFGILIKRRQQKIRKYTMRRLLQETELVEPLTPSGAMPNQAQMRILKETELRKVKVLGSGAFGTVYKGIWIPDGENVKIPVAIKVLRENTSPKANKEILDEAYVMAGVGSPYVSRLLGICLTSTVQLVTQLMPYGCLLDHVRENRGRLGSQDLLNWCMQIAKGMSYLEDVRLVHRDLAARNVLVKSPNHVKITDFGLARLLDIDETEYHADGGKVPIKWMALESILRRRFTHQSDVWSYGVTVWELMTFGAKPYDGIPAREIPDLLEKGERLPQPPICTIDVYMIMVKCWMIDSECRPRFRELVSEFSRMARDPQRFVVIQNEDLGPASP

In [ ]:
#| export
@lru_cache()
def get_uniprot_features(uniprot_id):
    "Given uniprot_id, get specific region for uniprot features."
    # uniprot REST API
    url = f"https://rest.uniprot.org/uniprotkb/{uniprot_id}.json"
    response = requests.get(url)

    if response.status_code == 200:
        data = response.json()
        # Extract the "features" section which contains information
        features = data.get('features', [])
        return features
    else:
        raise ValueError(f"Failed to retrieve UniProt features for {uniprot_id}")

In [ ]:
get_uniprot_features('P04626')[:3]

[{'type': 'Signal',
  'location': {'start': {'value': 1, 'modifier': 'EXACT'},
   'end': {'value': 22, 'modifier': 'EXACT'}},
  'description': '',
  'evidences': [{'evidenceCode': 'ECO:0000255'}]},
 {'type': 'Chain',
  'location': {'start': {'value': 23, 'modifier': 'EXACT'},
   'end': {'value': 1255, 'modifier': 'EXACT'}},
  'description': 'Receptor tyrosine-protein kinase erbB-2',
  'featureId': 'PRO_0000016669'},
 {'type': 'Topological domain',
  'location': {'start': {'value': 23, 'modifier': 'EXACT'},
   'end': {'value': 652, 'modifier': 'EXACT'}},
  'description': 'Extracellular',
  'evidences': [{'evidenceCode': 'ECO:0000255'}]}]

In [ ]:
#| export
def get_uniprot_kd(uniprot_id):
    "Get kinase domain sequences based on UniProt ID."
    features = get_uniprot_features(uniprot_id)
    out_regions = []
    seq = get_uniprot_seq(uniprot_id)

    for feature in features:
        if feature.get("type") == "Domain" and "Protein kinase" in feature.get("description", ""):
            start = feature['location']['start']['value']
            end = feature['location']['end']['value']
            region = {
                'uniprot_id': uniprot_id,
                'type': feature['type'],
                'start': start,
                'end': end,
                'description': feature['description'],
                'sequence': seq[start-1:end]
            }
            out_regions.append(region)

    return out_regions

In [ ]:
get_uniprot_kd('P04626')

[{'uniprot_id': 'P04626',
  'type': 'Domain',
  'start': 720,
  'end': 987,
  'description': 'Protein kinase',
  'sequence': 'LRKVKVLGSGAFGTVYKGIWIPDGENVKIPVAIKVLRENTSPKANKEILDEAYVMAGVGSPYVSRLLGICLTSTVQLVTQLMPYGCLLDHVRENRGRLGSQDLLNWCMQIAKGMSYLEDVRLVHRDLAARNVLVKSPNHVKITDFGLARLLDIDETEYHADGGKVPIKWMALESILRRRFTHQSDVWSYGVTVWELMTFGAKPYDGIPAREIPDLLEKGERLPQPPICTIDVYMIMVKCWMIDSECRPRFRELVSEFSRMARDPQRFV'}]

In [ ]:
#| export
def get_uniprot_type(uniprot_id,type_='Signal'):
    "Get region sequences based on UniProt ID features."
    features = get_uniprot_features(uniprot_id)
    out_regions = []
    seq = get_uniprot_seq(uniprot_id)

    for feature in features:
        if feature.get("type") == type_:
            start = feature['location']['start']['value']
            end = feature['location']['end']['value']
            region = {
                'uniprot_id': uniprot_id,
                'type': feature['type'],
                'start': start,
                'end': end,
                'description': feature['description'],
                'sequence': seq[start-1:end]
            }
            out_regions.append(region)

    return out_regions

In [ ]:
get_uniprot_type('P04626','Signal') # signal peptide

[{'uniprot_id': 'P04626',
  'type': 'Signal',
  'start': 1,
  'end': 22,
  'description': '',
  'sequence': 'MELAALCRWGLLLALLPPGAAS'}]

In [ ]:
get_uniprot_type('P04626','Transmembrane') # tm domain

[{'uniprot_id': 'P04626',
  'type': 'Transmembrane',
  'start': 653,
  'end': 675,
  'description': 'Helical',
  'sequence': 'SIISAVVGILLVVVLGVVFGILI'}]

## Mutate sequence

In [ ]:
#| export
def mutate(seq, # protein sequence
           *mutations, # e.g., E709A
           verbose=True,
           ):
    "Apply mutations to a protein sequence."
    seq_list = list(seq)  # convert to list for mutability
    
    for mut in mutations:
        # check mutation format
        if len(mut) < 3: raise ValueError(f"Invalid mutation format: {mut}")
        
        from_aa,pos,to_aa = mut[0],int(mut[1:-1])-1,mut[-1]

        # make sure position is within the sequence length
        if pos < 0 or pos >= len(seq_list): raise IndexError(f"Position {pos + 1} out of range for sequence length {len(seq_list)}")
        # make sure aa from mutations matches the residue on the sequence
        if seq_list[pos] != from_aa: raise ValueError(f"Expected {from_aa} at position {pos + 1}, found {seq_list[pos]}")
        
        seq_list[pos] = to_aa
        if verbose: print('Converted:', mut)
        
    return ''.join(seq_list)

In [ ]:
seq = get_uniprot_seq('P04626')
mut_seq = mutate(seq,'M1A','E2S')
mut_seq

Converted: M1A
Converted: E2S


'ASLAALCRWGLLLALLPPGAASTQVCTGTDMKLRLPASPETHLDMLRHLYQGCQVVQGNLELTYLPTNASLSFLQDIQEVQGYVLIAHNQVRQVPLQRLRIVRGTQLFEDNYALAVLDNGDPLNNTTPVTGASPGGLRELQLRSLTEILKGGVLIQRNPQLCYQDTILWKDIFHKNNQLALTLIDTNRSRACHPCSPMCKGSRCWGESSEDCQSLTRTVCAGGCARCKGPLPTDCCHEQCAAGCTGPKHSDCLACLHFNHSGICELHCPALVTYNTDTFESMPNPEGRYTFGASCVTACPYNYLSTDVGSCTLVCPLHNQEVTAEDGTQRCEKCSKPCARVCYGLGMEHLREVRAVTSANIQEFAGCKKIFGSLAFLPESFDGDPASNTAPLQPEQLQVFETLEEITGYLYISAWPDSLPDLSVFQNLQVIRGRILHNGAYSLTLQGLGISWLGLRSLRELGSGLALIHHNTHLCFVHTVPWDQLFRNPHQALLHTANRPEDECVGEGLACHQLCARGHCWGPGPTQCVNCSQFLRGQECVEECRVLQGLPREYVNARHCLPCHPECQPQNGSVTCFGPEADQCVACAHYKDPPFCVARCPSGVKPDLSYMPIWKFPDEEGACQPCPINCTHSCVDLDDKGCPAEQRASPLTSIISAVVGILLVVVLGVVFGILIKRRQQKIRKYTMRRLLQETELVEPLTPSGAMPNQAQMRILKETELRKVKVLGSGAFGTVYKGIWIPDGENVKIPVAIKVLRENTSPKANKEILDEAYVMAGVGSPYVSRLLGICLTSTVQLVTQLMPYGCLLDHVRENRGRLGSQDLLNWCMQIAKGMSYLEDVRLVHRDLAARNVLVKSPNHVKITDFGLARLLDIDETEYHADGGKVPIKWMALESILRRRFTHQSDVWSYGVTVWELMTFGAKPYDGIPAREIPDLLEKGERLPQPPICTIDVYMIMVKCWMIDSECRPRFRELVSEFSRMARDPQRFVVIQNEDLGPASP

In [ ]:
#| export
def compare_seq(original_seq, mutated_seq):
    "Compare original and mutated sequences."
    
    if len(original_seq) != len(mutated_seq): raise ValueError("Sequences must be the same length to compare.")

    differences = []
    for i, (orig, mut) in enumerate(zip(original_seq, mutated_seq), start=1):
        if orig != mut:
            differences.append((i, orig, mut))

    if not differences: print("No differences found. Sequences are identical.")
    else:
        print("Differences found at positions:")
        for pos, orig, mut in differences:
            print(f"  Position {pos}: {orig} → {mut}")

In [ ]:
compare_seq(seq,mut_seq)

Differences found at positions:
  Position 1: M → A
  Position 2: E → S


## End

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()